In [1]:
import httpx

response = httpx.get("http://localhost:11434/api/tags", timeout=5)
models = [m["name"] for m in response.json().get("models", [])]
print(f"Ollama is running. Installed models: {models}")

Ollama is running. Installed models: ['qwen2.5:3b-instruct', 'deepseek-r1:1.5b', 'qwen3.5:9b', 'qwen3.5:0.8b', 'llama3.2:3b', 'gemma3:1b', 'gemma3:270m']


In [2]:
def get_current_weather(city, unit = "celsius") -> str:
    return f"It is 23°{unit[0].upper()} and sunny in {city}." # no real API call to stay offline-friendly

SYSTEM_PROMPT = (
    "You are an assistant that can call tools. "
    "When the user asks something requiring fresh data, respond **only** with a JSON like: "
    'TOOL_CALL:{"name": <tool_name>, "args": { ... }}.'
)

TOOLS_SPEC = """
You can call exactly one tool:
- name: get_current_weather
  description: Return the current weather for a city.
  arguments:
    city: string
    unit: "celsius" | "fahrenheit"  (optional, default "celsius")
"""

# USER_QUESTION = "What is your name?"
USER_QUESTION = "What is the weather in San Diego today?"

In [3]:
from openai import OpenAI

client = OpenAI(api_key = "ollama", base_url = "http://localhost:11434/v1")

# MODEL = "gemma3:1b"
MODEL = "llama3.2:3b"

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT + "\n\n" + TOOLS_SPEC},
        {"role": "user", "content": USER_QUESTION},
    ],
    temperature=0,
)

output = response.choices[0].message.content
print("\n Model output:\n", output)


 Model output:
 TOOL_CALL:{"name": "get_current_weather", "args": {"city": "San Diego", "unit": "celsius"}}


In [4]:
prompt = 'You are part of the BriefBox email triage pipeline.\nReturn strict JSON only. Do not wrap the JSON in markdown.\nStage: classify\nInstruction: Classify the email into a category and confidence.\nJSON schema keys you may use: category, confidence, summary, entities, priority_score, score_rationale, suggested_lane, action, action_deadline, unsubscribe_candidate.\nSender: nina@boardops.example\nSubject: Approve the board packet by 2026-03-25\nBody: Action required. Please review the attached board packet and send approval by 2026-03-25.\nHeaders: {}'


response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt},
    ],
    temperature=0.7,
)

output = response.choices[0].message.content
print("\n Model output:\n", output)


 Model output:
 {"category":"approve","confidence":0.9,"summary":"Approve the board packet by deadline","entities":{},"priority_score":3,"score_rationale":"Deadline is in 1 month","suggested_lane":"board_ops","action":"Review and approve the board packet","action_deadline":1646115200,"unsubscribe_candidate":false}
